# scKITE Cell-Type Annotation Benchmark

本 notebook 保留原有 `linear_probe` / `full_finetune`，并加入冻结表示的 **5-NN zero-shot**。

默认采用与监督模式相同的数据边界：

- 原 `train` split 作为带标签 reference；
- 原 `test` split 作为 query；
- `val` 默认不参与 5-NN；
- label space 与监督模式一致，仅保留 train 中出现过的 cell types；
- 冻结预训练 encoder，只提取 `<cls>` cell embedding；
- 使用 uniform-weighted 5-nearest-neighbor classifier 和欧氏距离；
- 不训练分类头、不反向传播、不选择下游 checkpoint。

默认配置：

```python
cfg.evaluation_mode = "zero_shot_knn"
cfg.zero_shot_split_mode = "existing_split"
cfg.zero_shot_reference_split = "train"
cfg.zero_shot_query_split = "test"
cfg.zero_shot_use_val_as_reference = False
```

仍保留合并数据后重新分层划分的旧方案：

```python
cfg.zero_shot_split_mode = "resplit_all"
```

切换回监督评测：

```python
cfg.evaluation_mode = "supervised"
cfg.finetune_mode = "linear_probe"  # 或 full_finetune
```


In [ ]:




from __future__ import annotations

import json
import os
import random
import time
import warnings
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

try:
    from streaming import StreamingDataset
except Exception as exc:
    StreamingDataset = None
    streaming_import_error = exc


In [ ]:




REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'sckite').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'sckite').is_dir():
    raise FileNotFoundError('Run this notebook from inside the scKITE repository.')


@dataclass
class CFG:





    evaluation_mode: str = "zero_shot_knn"




    encoder_ckpt: str = str(REPO_ROOT / 'checkpoints' / 'stage2' / 'best.pt')
    model_name: str = "scKITE_stage2"




    data_root: str = str(REPO_ROOT / 'data' / 'cell_type_annotation' / 'Cross_tissue_immune')
    train_dir: str = "train"
    val_dir: str = "val"
    test_dir: str = "test"




    output_root: str = str(REPO_ROOT / 'outputs' / 'cell_type_annotation')
    output_dir: str = ""
    run_name: str = ""




    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    amp: bool = True




    gene_id_min: int = 3
    gene_id_max: int = 62712
    pad_token_id: int = 0
    cls_token_id: int = 1
    pad_value: float = -2.0
    cls_value: float = -1.0




    max_len: int = 2049
    value_mode: str = "rank_binned"
    n_bins: int = 51
    add_cls_if_missing: bool = True




    batch_size: int = 18
    num_workers: int = 8










    zero_shot_split_mode: str = "existing_split"
    zero_shot_reference_split: str = "train"
    zero_shot_query_split: str = "test"
    zero_shot_use_val_as_reference: bool = False


    zero_shot_reference_ratio: float = 0.5
    zero_shot_split_seed: int = 42
    zero_shot_stratify: bool = True
    zero_shot_pool_splits: Tuple[str, ...] = ("train", "val", "test")

    zero_shot_k: int = 5


    zero_shot_weights: str = "uniform"
    zero_shot_metric: str = "minkowski"
    zero_shot_p: int = 2


    zero_shot_manifest_path: str = ""
    zero_shot_reuse_manifest: bool = True




    epochs: int = 10
    patience: int = 5
    finetune_mode: str = "full_finetune"
    head_type: str = "mlp"
    lr_encoder: float = 1e-5
    lr_head: float = 1e-3
    weight_decay: float = 1e-4
    dropout: float = 0.1
    metric_for_best: str = "macro_f1"
    select_ckpt_for_test: str = "loss"




    nhead: int = 8
    activation: str = "gelu"




    save_cell_emb: bool = True
    make_umap: bool = True
    umap_max_cells: int = 200000

    umap_top_n_cell_types: Optional[int] = None
    confusion_matrix_top_n_cell_types: Optional[int] = 20

    umap_n_neighbors: int = 30
    umap_min_dist: float = 0.40
    umap_point_size: float = 2.2
    umap_point_alpha: float = 0.85
    umap_show_panel_titles: bool = False
    umap_show_legend: bool = True
    umap_show_suptitle: bool = False
    umap_show_center_divider: bool = False
    confusion_matrix_show_values: bool = False

    redraw_saved_embeddings: bool = True
    saved_embeddings_dir: str = ""
    saved_embedding_path: str = ""
    saved_predictions_path: str = ""
    saved_plot_output_dir: str = ""
    saved_plot_prefix: str = "saved_embedding"
    preview_samples_per_split: int = 2000


cfg = CFG()


def set_global_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))


set_global_seed(cfg.seed)

cfg.evaluation_mode = str(cfg.evaluation_mode).strip().lower()
cfg.zero_shot_split_mode = str(cfg.zero_shot_split_mode).strip().lower()
cfg.zero_shot_reference_split = str(cfg.zero_shot_reference_split).strip().lower()
cfg.zero_shot_query_split = str(cfg.zero_shot_query_split).strip().lower()
cfg.finetune_mode = str(cfg.finetune_mode).strip().lower()
cfg.head_type = str(cfg.head_type).strip().lower()
cfg.metric_for_best = str(cfg.metric_for_best).strip().lower()
cfg.select_ckpt_for_test = str(cfg.select_ckpt_for_test).strip().lower()

if cfg.evaluation_mode not in {"zero_shot_knn", "supervised"}:
    raise ValueError("evaluation_mode 仅支持 zero_shot_knn 或 supervised。")

if cfg.zero_shot_split_mode not in {"existing_split", "resplit_all"}:
    raise ValueError(
        "zero_shot_split_mode 仅支持 existing_split 或 resplit_all。"
    )
if cfg.zero_shot_reference_split not in {"train", "val", "test"}:
    raise ValueError("zero_shot_reference_split 必须是 train/val/test。")
if cfg.zero_shot_query_split not in {"train", "val", "test"}:
    raise ValueError("zero_shot_query_split 必须是 train/val/test。")
if cfg.zero_shot_reference_split == cfg.zero_shot_query_split:
    raise ValueError("reference split 与 query split 不能相同。")

if cfg.finetune_mode not in {"linear_probe", "full_finetune"}:
    raise ValueError("finetune_mode 仅支持 linear_probe 或 full_finetune。")

if cfg.head_type not in {"strict_linear", "linear", "mlp"}:
    raise ValueError("head_type 仅支持 strict_linear、linear 或 mlp。")

if cfg.metric_for_best not in {
    "accuracy", "micro_f1", "macro_f1", "precision", "recall", "weighted_f1"
}:
    raise ValueError("metric_for_best 配置无效。")

if cfg.select_ckpt_for_test not in {"perf", "loss", "last"}:
    raise ValueError("select_ckpt_for_test 仅支持 perf、loss 或 last。")

if not 0.0 < float(cfg.zero_shot_reference_ratio) < 1.0:
    raise ValueError("zero_shot_reference_ratio 必须位于 (0, 1)。")

if int(cfg.zero_shot_k) <= 0:
    raise ValueError("zero_shot_k 必须为正整数。")


dataset_name = Path(cfg.data_root.rstrip("/")).name
cfg.model_name = str(cfg.model_name).strip() or Path(cfg.encoder_ckpt).stem


if cfg.evaluation_mode == "zero_shot_knn":
    if cfg.zero_shot_split_mode == "existing_split":
        reference_name = cfg.zero_shot_reference_split
        if cfg.zero_shot_use_val_as_reference and reference_name != "val":
            reference_name = f"{reference_name}_plus_val"
        protocol_name = (
            f"5nn_existing_{reference_name}_reference_"
            f"{cfg.zero_shot_query_split}_query"
        )
    else:
        protocol_name = (
            f"5nn_resplit_stratified_ref"
            f"{int(round(cfg.zero_shot_reference_ratio * 100))}_"
            f"seed{cfg.zero_shot_split_seed}"
        )

    cfg.output_dir = os.path.join(
        cfg.output_root,
        dataset_name,
        cfg.model_name,
        "zero_shot_knn",
        protocol_name,
    )
    cfg.run_name = "__".join(
        [dataset_name, cfg.model_name, "zero_shot_knn", protocol_name]
    )
else:
    cfg.output_dir = os.path.join(
        cfg.output_root,
        dataset_name,
        cfg.model_name,
        cfg.finetune_mode,
        cfg.head_type,
    )
    cfg.run_name = "__".join(
        [dataset_name, cfg.model_name, cfg.finetune_mode, cfg.head_type]
    )

os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(os.path.join(cfg.output_dir, "figures"), exist_ok=True)


if not str(cfg.zero_shot_manifest_path).strip():
    shared_split_dir = os.path.join(
        cfg.output_root,
        dataset_name,
        "_shared_splits",
    )
    os.makedirs(shared_split_dir, exist_ok=True)
    if cfg.zero_shot_split_mode == "existing_split":
        reference_name = cfg.zero_shot_reference_split
        if cfg.zero_shot_use_val_as_reference and reference_name != "val":
            reference_name = f"{reference_name}_plus_val"
        manifest_filename = (
            f"existing_{reference_name}_reference_"
            f"{cfg.zero_shot_query_split}_query.csv"
        )
    else:
        manifest_filename = (
            f"resplit_ref{int(round(cfg.zero_shot_reference_ratio * 100))}_"
            f"stratified_seed{cfg.zero_shot_split_seed}.csv"
        )

    cfg.zero_shot_manifest_path = os.path.join(
        shared_split_dir,
        manifest_filename,
    )

print("evaluation_mode :", cfg.evaluation_mode)
print("dataset_name    :", dataset_name)
print("model_name      :", cfg.model_name)
print("run_name        :", cfg.run_name)
print("output_dir      :", cfg.output_dir)
print("device          :", cfg.device)
if cfg.evaluation_mode == "zero_shot_knn":
    print("zero-shot mode  :", cfg.zero_shot_split_mode)
    print("reference split :", cfg.zero_shot_reference_split)
    print("query split     :", cfg.zero_shot_query_split)
    print("val as reference:", cfg.zero_shot_use_val_as_reference)
    print("shared manifest :", cfg.zero_shot_manifest_path)


In [ ]:




def load_checkpoint_state(path: str) -> Tuple[Dict[str, torch.Tensor], Dict[str, Any]]:
    ckpt = torch.load(path, map_location="cpu")

    if isinstance(ckpt, dict) and "encoder_state_dict" in ckpt:
        state = ckpt["encoder_state_dict"]
        meta_cfg = ckpt.get("config", ckpt.get("cfg", {}))
    elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
        meta_cfg = ckpt.get("config", ckpt.get("cfg", {}))
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state = ckpt["state_dict"]
        meta_cfg = ckpt.get("config", ckpt.get("cfg", {}))
    elif isinstance(ckpt, dict):
        state = ckpt
        meta_cfg = {}
    else:
        raise TypeError("无法解析 checkpoint 格式。")

    if not isinstance(state, dict):
        raise TypeError("checkpoint state_dict 不是字典。")

    return dict(state), dict(meta_cfg or {})


def normalize_encoder_state_keys(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    """兼容 canonical encoder、DDP 和下游 AnnotationModel checkpoint。"""
    normalized = {}
    for key, value in state.items():
        new_key = str(key)
        while new_key.startswith("module."):
            new_key = new_key[len("module."):]
        normalized[new_key] = value


    if (
        "shared_token_embedding.weight" not in normalized
        and "encoder.shared_token_embedding.weight" in normalized
    ):
        normalized = {
            (key[len("encoder."):] if key.startswith("encoder.") else key): value
            for key, value in normalized.items()
        }


    if (
        "shared_token_embedding.weight" not in normalized
        and "model.encoder.shared_token_embedding.weight" in normalized
    ):
        normalized = {
            (
                key[len("model.encoder."):]
                if key.startswith("model.encoder.")
                else key
            ): value
            for key, value in normalized.items()
        }

    return normalized


def filter_encoder_state(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    keep_prefixes = (
        "shared_token_embedding.",
        "token_norm.",
        "value_encoder.",
        "mask_flag_embedding.",
        "encoder.",
    )
    return {
        key: value
        for key, value in state.items()
        if key.startswith(keep_prefixes)
    }


def infer_encoder_config(
    state: Dict[str, torch.Tensor],
    meta_cfg: Dict[str, Any],
    cfg: CFG,
) -> Dict[str, Any]:
    if "shared_token_embedding.weight" not in state:
        raise KeyError(
            "规范化后的 checkpoint 中缺少 shared_token_embedding.weight。"
        )

    vocab_size, hidden_dim = state["shared_token_embedding.weight"].shape

    layer_ids: List[int] = []
    for key in state:
        if key.startswith("encoder.layers."):
            parts = key.split(".")
            if len(parts) > 2 and parts[2].isdigit():
                layer_ids.append(int(parts[2]))

    nlayers = (
        max(layer_ids) + 1
        if layer_ids
        else int(meta_cfg.get("nlayers", 0))
    )

    if nlayers <= 0:
        raise ValueError("无法从 checkpoint 推断 encoder 层数。")

    d_hid = (
        int(state["encoder.layers.0.ffn.0.weight"].shape[0])
        if "encoder.layers.0.ffn.0.weight" in state
        else int(meta_cfg.get("d_hid", int(hidden_dim) * 4))
    )

    value_hidden_dim = (
        int(state["value_encoder.proj.0.weight"].shape[0])
        if "value_encoder.proj.0.weight" in state
        else int(meta_cfg.get("value_hidden_dim", 128))
    )

    return {
        "vocab_size": int(vocab_size),
        "hidden_dim": int(hidden_dim),
        "nlayers": int(nlayers),
        "d_hid": int(d_hid),
        "value_hidden_dim": int(value_hidden_dim),
        "nhead": int(meta_cfg.get("nhead", cfg.nhead)),
        "activation": str(meta_cfg.get("activation", cfg.activation)),
        "stage": str(meta_cfg.get("stage", meta_cfg.get("model_stage", "unknown"))),
    }


if cfg.redraw_saved_embeddings:
    encoder_state: Dict[str, torch.Tensor] = {}
    enc_cfg: Dict[str, Any] = {}
    print("[redraw mode] 跳过 checkpoint 加载与模型结构推断。")
else:
    raw_encoder_state, encoder_meta = load_checkpoint_state(cfg.encoder_ckpt)
    normalized_encoder_state = normalize_encoder_state_keys(raw_encoder_state)
    encoder_state = filter_encoder_state(normalized_encoder_state)
    enc_cfg = infer_encoder_config(encoder_state, encoder_meta, cfg)

    cfg.vocab_size = enc_cfg["vocab_size"]
    cfg.hidden_dim = enc_cfg["hidden_dim"]
    cfg.nlayers = enc_cfg["nlayers"]
    cfg.d_hid = enc_cfg["d_hid"]
    cfg.value_hidden_dim = enc_cfg["value_hidden_dim"]
    cfg.nhead = enc_cfg["nhead"]
    cfg.activation = enc_cfg["activation"]

    with open(os.path.join(cfg.output_dir, "encoder_config.json"), "w", encoding="utf-8") as handle:
        json.dump(enc_cfg, handle, ensure_ascii=False, indent=2)

    with open(os.path.join(cfg.output_dir, "run_config.json"), "w", encoding="utf-8") as handle:
        json.dump(asdict(cfg), handle, ensure_ascii=False, indent=2)

    print(json.dumps(enc_cfg, ensure_ascii=False, indent=2))


In [ ]:




CELL_TYPE_KEYS = (
    "cell_type",
    "celltype",
    "celltype_label",
    "label",
    "annotation",
    "free_annotation",
)

GENE_KEYS = (
    "genes",
    "gene_ids",
    "input_ids",
    "gene",
)

EXPRESSION_KEYS = (
    "expressions",
    "expr",
    "values",
    "expression",
    "expr_values",
)

METADATA_KEYS = (
    "donor",
    "method",
    "organ_tissue",
    "cluster_label",
    "compartment",
    "gender",
)


def first_present(
    sample: Dict[str, Any],
    keys: Sequence[str],
    required_name: str,
) -> Any:
    for key in keys:
        if key in sample:
            return sample[key]
    raise KeyError(
        f"样本缺少 {required_name} 字段；候选字段={list(keys)}；"
        f"实际字段={list(sample.keys())}"
    )


def get_cell_type_only(sample: Dict[str, Any]) -> str:
    return str(first_present(sample, CELL_TYPE_KEYS, "cell type"))


def get_cell_id_only(sample: Dict[str, Any], raw_index: int) -> str:
    return str(sample.get("cell_id", sample.get("id", raw_index)))


def normalize_sample(
    sample: Dict[str, Any],
    raw_index: int,
    source_split: str,
) -> Dict[str, Any]:
    genes = np.asarray(
        first_present(sample, GENE_KEYS, "genes"),
        dtype=np.int64,
    ).reshape(-1)
    expressions = np.asarray(
        first_present(sample, EXPRESSION_KEYS, "expressions"),
        dtype=np.float32,
    ).reshape(-1)

    if len(genes) != len(expressions):
        raise ValueError(
            f"{source_split}:{raw_index} genes/expressions 长度不一致："
            f"{len(genes)} vs {len(expressions)}"
        )

    cell_type = get_cell_type_only(sample)
    cell_id = get_cell_id_only(sample, raw_index)
    sample_uid = f"{source_split}:{raw_index}"

    return {
        "sample_uid": sample_uid,
        "source_split": str(source_split),
        "raw_mds_index": int(raw_index),
        "cell_id": cell_id,
        "cell_type": cell_type,
        "genes": genes,
        "expressions": expressions,
        "metadata": {key: sample.get(key, "") for key in METADATA_KEYS},
    }


def open_mds_split(split_dir: str, split_name: str) -> StreamingDataset:
    if StreamingDataset is None:
        raise ImportError(
            "无法导入 streaming.StreamingDataset，请先安装 mosaicml-streaming。"
        ) from streaming_import_error
    split_path = Path(split_dir)
    if not split_path.exists():
        raise FileNotFoundError(f"{split_name} MDS 目录不存在：{split_path}")

    dataset = StreamingDataset(
        local=str(split_path),
        remote=None,
        shuffle=False,
    )

    if len(dataset) == 0:
        raise RuntimeError(f"{split_name} MDS 为空：{split_path}")

    print(f"[MDS lazy open] {split_name}: path={split_path}, n={len(dataset):,}")
    return dataset


def scan_label_records(
    mds: StreamingDataset,
    split_name: str,
) -> Tuple[Counter, pd.DataFrame]:
    """只缓存 cell_id、label 和索引，不缓存 genes/expressions。"""
    counts: Counter = Counter()
    rows: List[Dict[str, Any]] = []

    for raw_index in range(len(mds)):
        sample = dict(mds[raw_index])
        cell_type = get_cell_type_only(sample)
        cell_id = get_cell_id_only(sample, raw_index)
        counts[cell_type] += 1
        rows.append({
            "sample_uid": f"{split_name}:{raw_index}",
            "source_split": split_name,
            "raw_mds_index": int(raw_index),
            "cell_id": cell_id,
            "cell_type": cell_type,
        })

    print(
        f"[label scan] {split_name}: n={len(mds):,}, "
        f"n_labels={len(counts)}"
    )
    return counts, pd.DataFrame(rows)


mds_by_split: Dict[str, StreamingDataset] = {}
if not cfg.redraw_saved_embeddings:
    mds_by_split = {
        "train": open_mds_split(
            os.path.join(cfg.data_root, cfg.train_dir),
            "train",
        ),
        "val": open_mds_split(
            os.path.join(cfg.data_root, cfg.val_dir),
            "val",
        ),
        "test": open_mds_split(
            os.path.join(cfg.data_root, cfg.test_dir),
            "test",
        ),
    }

label_counts_by_split: Dict[str, Counter] = {}
label_records_by_split: Dict[str, pd.DataFrame] = {}

for split_name, mds_split in mds_by_split.items():
    counts, records = scan_label_records(mds_split, split_name)
    label_counts_by_split[split_name] = counts
    label_records_by_split[split_name] = records

if cfg.redraw_saved_embeddings:
    label_names: List[str] = []
    pooled_label_records = pd.DataFrame()
    valid_indices_by_split: Dict[str, Optional[List[int]]] = {}
    evaluation_setting = "redraw_saved_embeddings"
elif (
    cfg.evaluation_mode == "zero_shot_knn"
    and cfg.zero_shot_split_mode == "resplit_all"
):
    missing_splits = [
        split_name
        for split_name in cfg.zero_shot_pool_splits
        if split_name not in mds_by_split
    ]
    if missing_splits:
        raise KeyError(
            f"zero_shot_pool_splits 中存在未知 split：{missing_splits}"
        )

    pooled_label_records = pd.concat(
        [label_records_by_split[name] for name in cfg.zero_shot_pool_splits],
        ignore_index=True,
    )
    label_names = sorted(
        pooled_label_records["cell_type"].astype(str).unique()
    )
    valid_indices_by_split = {
        split_name: None
        for split_name in cfg.zero_shot_pool_splits
    }
    evaluation_setting = "resplit_all_stratified_5nn_zero_shot"
else:

    train_labels = set(label_counts_by_split["train"].keys())
    label_names = sorted(train_labels)
    pooled_label_records = pd.DataFrame()
    valid_indices_by_split = {"train": None}

    for split_name in ("val", "test"):
        records = label_records_by_split[split_name]
        keep_mask = records["cell_type"].isin(train_labels)
        valid_indices_by_split[split_name] = (
            records.loc[keep_mask, "raw_mds_index"].astype(int).tolist()
        )
        unknown_counts = Counter(
            records.loc[~keep_mask, "cell_type"].tolist()
        )
        if unknown_counts:
            print(
                f"[{split_name}] removed train-unseen labels:",
                dict(unknown_counts),
            )

    evaluation_setting = (
        "existing_train_reference_test_query_5nn_zero_shot"
        if cfg.evaluation_mode == "zero_shot_knn"
        else "closed_set_train_seen_cell_types_only"
    )

label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}
n_cls = len(label2id)

if n_cls == 0 and not cfg.redraw_saved_embeddings:
    raise RuntimeError("没有可用的 cell type 标签。")

if not cfg.redraw_saved_embeddings:
    with open(os.path.join(cfg.output_dir, "label2id.json"), "w", encoding="utf-8") as handle:
        json.dump(label2id, handle, ensure_ascii=False, indent=2)

    with open(os.path.join(cfg.output_dir, "id2label.json"), "w", encoding="utf-8") as handle:
        json.dump({str(k): v for k, v in id2label.items()}, handle, ensure_ascii=False, indent=2)

    stats = {
        "evaluation_setting": evaluation_setting,
        "n_cell_types": int(n_cls),
        "label_names": label_names,
        "split_sizes": {name: int(len(ds)) for name, ds in mds_by_split.items()},
        "label_counts": {
            name: dict(counts)
            for name, counts in label_counts_by_split.items()
        },
    }

    with open(os.path.join(cfg.output_dir, "lazy_dataset_stats.json"), "w", encoding="utf-8") as handle:
        json.dump(stats, handle, ensure_ascii=False, indent=2)

print("n_cls:", n_cls)


In [ ]:




def rank_binning(expr: np.ndarray, n_bins: int) -> np.ndarray:
    expr = np.asarray(expr, dtype=np.float32).reshape(-1)
    expr = np.nan_to_num(expr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    output = np.zeros_like(expr, dtype=np.float32)
    nonzero = expr > 0
    if nonzero.sum() == 0:
        return output

    values = expr[nonzero]
    if np.all(values == values[0]):
        output[nonzero] = 1.0
        return output

    edges = np.unique(
        np.quantile(
            values,
            np.linspace(0, 1, max(int(n_bins) - 1, 2)),
        )
    )
    bins = np.searchsorted(edges, values, side="right")
    output[nonzero] = np.clip(bins, 1, int(n_bins) - 1).astype(np.float32)
    return output


def process_expr(expr: np.ndarray, cfg: CFG) -> np.ndarray:
    expr = np.asarray(expr, dtype=np.float32).reshape(-1)
    expr = np.nan_to_num(expr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    mode = str(cfg.value_mode).strip().lower()

    if mode == "rank_binned":
        return rank_binning(expr, cfg.n_bins)
    if mode == "log1p_if_raw":
        max_value = float(np.max(expr)) if expr.size else 0.0
        if max_value > 30.0:
            return np.log1p(np.clip(expr, 0.0, None)).astype(np.float32)
        return expr.astype(np.float32, copy=False)
    if mode == "as_is":
        return expr.astype(np.float32, copy=False)

    raise ValueError(f"未知 value_mode：{cfg.value_mode}")


def validate_tokens(
    genes: np.ndarray,
    source_split: str,
    raw_index: int,
    cfg: CFG,
) -> None:
    genes = np.asarray(genes, dtype=np.int64)

    invalid_range = (genes < 0) | (genes >= cfg.vocab_size)
    if invalid_range.any():
        bad = genes[invalid_range][:20].tolist()
        raise ValueError(f"{source_split}:{raw_index} 存在越界 token：{bad}")

    allowed = (
        (genes == cfg.pad_token_id)
        | (genes == cfg.cls_token_id)
        | ((genes >= cfg.gene_id_min) & (genes <= cfg.gene_id_max))
    )
    if (~allowed).any():
        bad = genes[~allowed][:20].tolist()
        raise ValueError(f"{source_split}:{raw_index} 含有非 gene-side token：{bad}")


class LazyMDSCellTypeDataset(Dataset):
    def __init__(
        self,
        mds: StreamingDataset,
        label2id: Dict[str, int],
        split_name: str,
        cfg: CFG,
        valid_indices: Optional[List[int]] = None,
    ) -> None:
        self.mds = mds
        self.label2id = dict(label2id)
        self.split_name = str(split_name)
        self.cfg = cfg
        self.valid_indices = valid_indices

        kept = len(self.mds) if valid_indices is None else len(valid_indices)
        print(f"{self.split_name}: lazy keep {kept:,}/{len(self.mds):,}")

    def __len__(self) -> int:
        return len(self.mds) if self.valid_indices is None else len(self.valid_indices)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        raw_index = (
            int(index)
            if self.valid_indices is None
            else int(self.valid_indices[index])
        )
        raw_sample = dict(self.mds[raw_index])
        sample = normalize_sample(raw_sample, raw_index, self.split_name)
        cell_type = str(sample["cell_type"])

        if cell_type not in self.label2id:
            raise KeyError(
                f"[{self.split_name}] cell type {cell_type!r} 不在 label2id 中。"
            )

        genes = np.asarray(sample["genes"], dtype=np.int64)
        raw_expr = np.asarray(sample["expressions"], dtype=np.float32)

        if len(genes) > 0 and int(genes[0]) == self.cfg.cls_token_id:
            expressions = np.empty_like(raw_expr, dtype=np.float32)
            expressions[0] = self.cfg.cls_value
            if len(raw_expr) > 1:
                expressions[1:] = process_expr(raw_expr[1:], self.cfg)
        else:
            expressions = process_expr(raw_expr, self.cfg)

        validate_tokens(genes, self.split_name, raw_index, self.cfg)

        return {
            **sample,
            "label": int(self.label2id[cell_type]),
            "genes": genes,
            "expressions": expressions,
        }


def collate_gene_expr(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    genes_list: List[torch.Tensor] = []
    expr_list: List[torch.Tensor] = []
    labels: List[int] = []
    cell_ids: List[str] = []
    cell_types: List[str] = []
    sample_uids: List[str] = []
    source_splits: List[str] = []
    raw_indices: List[int] = []
    metadata: List[Dict[str, Any]] = []

    for sample in samples:
        genes = np.asarray(sample["genes"], dtype=np.int64)
        expr = np.asarray(sample["expressions"], dtype=np.float32)

        if len(genes) != len(expr):
            raise ValueError(f"{sample['sample_uid']}: genes/expressions 长度不一致。")

        if cfg.add_cls_if_missing and not (
            len(genes) > 0 and int(genes[0]) == cfg.cls_token_id
        ):
            genes = np.concatenate([[cfg.cls_token_id], genes], axis=0)
            expr = np.concatenate([[cfg.cls_value], expr], axis=0).astype(np.float32)

        if len(genes) > cfg.max_len:
            genes = genes[:cfg.max_len]
            expr = expr[:cfg.max_len]

        genes_list.append(torch.as_tensor(genes, dtype=torch.long))
        expr_list.append(torch.as_tensor(expr, dtype=torch.float32))
        labels.append(int(sample["label"]))
        cell_ids.append(str(sample["cell_id"]))
        cell_types.append(str(sample["cell_type"]))
        sample_uids.append(str(sample["sample_uid"]))
        source_splits.append(str(sample["source_split"]))
        raw_indices.append(int(sample["raw_mds_index"]))
        metadata.append(dict(sample["metadata"]))

    batch_size = len(samples)
    max_length = min(cfg.max_len, max(tensor.numel() for tensor in genes_list))

    gene_ids = torch.full(
        (batch_size, max_length),
        cfg.pad_token_id,
        dtype=torch.long,
    )
    expr_values = torch.full(
        (batch_size, max_length),
        cfg.pad_value,
        dtype=torch.float32,
    )

    for row_index, (genes, expr) in enumerate(zip(genes_list, expr_list)):
        length = min(genes.numel(), max_length)
        gene_ids[row_index, :length] = genes[:length]
        expr_values[row_index, :length] = expr[:length]

    return {
        "gene_ids": gene_ids,
        "expr_values": expr_values,
        "mask_flags": torch.zeros_like(gene_ids),
        "src_key_padding_mask": gene_ids.eq(cfg.pad_token_id),
        "labels": torch.tensor(labels, dtype=torch.long),
        "cell_ids": cell_ids,
        "cell_types": cell_types,
        "sample_uids": sample_uids,
        "source_splits": source_splits,
        "raw_mds_indices": raw_indices,
        "metadata": metadata,
    }


def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        collate_fn=collate_gene_expr,
        pin_memory=cfg.device.startswith("cuda"),
        persistent_workers=(cfg.num_workers > 0),
    )


def check_dataset_preview(
    dataset: Dataset,
    split_name: str,
    max_samples: int,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for index in range(min(len(dataset), int(max_samples))):
        sample = dataset[index]
        genes = np.asarray(sample["genes"], dtype=np.int64)
        expr = np.asarray(sample["expressions"], dtype=np.float32)
        rows.append({
            "split": split_name,
            "sample_uid": sample["sample_uid"],
            "seq_len": int(len(genes)),
            "gene_max": int(np.max(genes)) if len(genes) else -1,
            "expr_max": float(np.max(expr)) if len(expr) else float("nan"),
            "has_cls_at_0": bool(len(genes) > 0 and genes[0] == cfg.cls_token_id),
            "cell_type": str(sample["cell_type"]),
        })
    return pd.DataFrame(rows)


if cfg.redraw_saved_embeddings:
    dataset_splits = []
elif cfg.evaluation_mode == "zero_shot_knn":
    if cfg.zero_shot_split_mode == "existing_split":
        dataset_splits = [cfg.zero_shot_reference_split]
        if (
            cfg.zero_shot_use_val_as_reference
            and "val" not in dataset_splits
            and cfg.zero_shot_query_split != "val"
        ):
            dataset_splits.append("val")
        if cfg.zero_shot_query_split not in dataset_splits:
            dataset_splits.append(cfg.zero_shot_query_split)
    else:
        dataset_splits = list(cfg.zero_shot_pool_splits)
else:
    dataset_splits = ["train", "val", "test"]


datasets: Dict[str, LazyMDSCellTypeDataset] = {}
for split_name in dataset_splits:
    datasets[split_name] = LazyMDSCellTypeDataset(
        mds=mds_by_split[split_name],
        label2id=label2id,
        split_name=split_name,
        cfg=cfg,
        valid_indices=valid_indices_by_split.get(split_name),
    )

preview_frames = [
    check_dataset_preview(
        datasets[split_name],
        split_name,
        cfg.preview_samples_per_split,
    )
    for split_name in dataset_splits
]
if preview_frames:
    preview_df = pd.concat(preview_frames, ignore_index=True)
    preview_df.to_csv(os.path.join(cfg.output_dir, "data_check_preview.csv"), index=False)
    print(preview_df.groupby("split")["seq_len"].describe())
    print(preview_df.groupby("split")["has_cls_at_0"].mean())
else:
    preview_df = pd.DataFrame()
    print("[redraw mode] 跳过 MDS Dataset/DataLoader 构建。")


In [ ]:




class ValueEncoder(nn.Module):
    def __init__(self, hidden_dim: int, value_hidden_dim: int, dropout: float):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(1, value_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(value_hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.proj(values.float().unsqueeze(-1))


class EncoderBlock(nn.Module):
    def __init__(
        self,
        hidden_dim: int,
        nhead: int,
        d_hid: int,
        dropout: float,
        activation: str,
    ) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.self_attn = nn.MultiheadAttention(
            hidden_dim,
            nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        activation_module = nn.GELU() if activation == "gelu" else nn.ReLU()
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, d_hid),
            activation_module,
            nn.Dropout(dropout),
            nn.Linear(d_hid, hidden_dim),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        hidden = self.norm1(x)
        attention_output, _ = self.self_attn(
            hidden,
            hidden,
            hidden,
            key_padding_mask=src_key_padding_mask,
            need_weights=False,
        )
        x = x + self.dropout(attention_output)
        hidden = self.norm2(x)
        return x + self.dropout(self.ffn(hidden))


class EncoderStack(nn.Module):
    def __init__(
        self,
        hidden_dim: int,
        nhead: int,
        d_hid: int,
        nlayers: int,
        dropout: float,
        activation: str,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(hidden_dim, nhead, d_hid, dropout, activation)
            for _ in range(nlayers)
        ])
        self.final_norm = nn.LayerNorm(hidden_dim)

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        return self.final_norm(x)


class MyEncoder(nn.Module):
    def __init__(self, cfg: CFG) -> None:
        super().__init__()
        self.shared_token_embedding = nn.Embedding(
            cfg.vocab_size,
            cfg.hidden_dim,
            padding_idx=cfg.pad_token_id,
        )
        self.token_norm = nn.LayerNorm(cfg.hidden_dim)
        self.value_encoder = ValueEncoder(
            cfg.hidden_dim,
            cfg.value_hidden_dim,
            cfg.dropout,
        )
        self.mask_flag_embedding = nn.Embedding(2, cfg.hidden_dim)
        self.encoder = EncoderStack(
            cfg.hidden_dim,
            cfg.nhead,
            cfg.d_hid,
            cfg.nlayers,
            cfg.dropout,
            cfg.activation,
        )

    def forward(
        self,
        gene_ids: torch.Tensor,
        expr_values: torch.Tensor,
        mask_flags: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        token_embedding = self.token_norm(self.shared_token_embedding(gene_ids))
        value_embedding = self.value_encoder(expr_values)
        mask_embedding = self.mask_flag_embedding(mask_flags.long())
        return self.encoder(
            token_embedding + value_embedding + mask_embedding,
            src_key_padding_mask=src_key_padding_mask,
        )


def load_encoder_weights(
    encoder: MyEncoder,
    state: Dict[str, torch.Tensor],
) -> None:
    required = {
        "shared_token_embedding.weight",
        "encoder.final_norm.weight",
        "encoder.final_norm.bias",
    }
    missing_required = sorted(required - set(state.keys()))
    if missing_required:
        raise KeyError(f"checkpoint 缺少关键 encoder 权重：{missing_required}")

    missing, unexpected = encoder.load_state_dict(state, strict=True)
    if missing or unexpected:
        raise RuntimeError(
            f"encoder strict load 失败；missing={missing}, unexpected={unexpected}"
        )
    print("[OK] encoder checkpoint 完整加载，包括 encoder.final_norm。")


class StrictLinearHead(nn.Module):
    def __init__(self, hidden_dim: int, n_classes: int) -> None:
        super().__init__()
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


class LinearHead(nn.Module):
    def __init__(self, hidden_dim: int, n_classes: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MLPHead(nn.Module):
    def __init__(self, hidden_dim: int, n_classes: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class FrozenEmbeddingModel(nn.Module):
    """Zero-shot 模式：只有冻结 encoder，不构建分类头。"""
    def __init__(self, cfg: CFG, state: Dict[str, torch.Tensor]) -> None:
        super().__init__()
        self.cfg = cfg
        self.encoder = MyEncoder(cfg)
        load_encoder_weights(self.encoder, state)
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False
        self.encoder.eval()

    @torch.no_grad()
    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        encoder_output = self.encoder(
            batch["gene_ids"].to(self.cfg.device),
            batch["expr_values"].to(self.cfg.device),
            batch["mask_flags"].to(self.cfg.device),
            batch["src_key_padding_mask"].to(self.cfg.device),
        )
        return encoder_output[:, 0, :]


class AnnotationModel(nn.Module):
    def __init__(
        self,
        cfg: CFG,
        state: Dict[str, torch.Tensor],
        n_classes: int,
    ) -> None:
        super().__init__()
        self.cfg = cfg
        self.encoder = MyEncoder(cfg)
        load_encoder_weights(self.encoder, state)

        if cfg.head_type == "strict_linear":
            self.classifier = StrictLinearHead(cfg.hidden_dim, n_classes)
        elif cfg.head_type == "linear":
            self.classifier = LinearHead(cfg.hidden_dim, n_classes, cfg.dropout)
        else:
            self.classifier = MLPHead(cfg.hidden_dim, n_classes, cfg.dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        encoder_output = self.encoder(
            batch["gene_ids"].to(self.cfg.device),
            batch["expr_values"].to(self.cfg.device),
            batch["mask_flags"].to(self.cfg.device),
            batch["src_key_padding_mask"].to(self.cfg.device),
        )
        cell_embedding = encoder_output[:, 0, :]
        logits = self.classifier(cell_embedding)
        return {"logits": logits, "cell_emb": cell_embedding}


In [ ]:




def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int,
) -> Dict[str, float]:
    labels = list(range(n_classes))
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "micro_f1": float(
            f1_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        ),
        "precision": float(
            precision_score(
                y_true, y_pred, labels=labels, average="macro", zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y_true, y_pred, labels=labels, average="macro", zero_division=0
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true, y_pred, labels=labels, average="weighted", zero_division=0
            )
        ),
    }


@torch.no_grad()
def extract_frozen_embeddings(
    model: FrozenEmbeddingModel,
    loader: DataLoader,
) -> Tuple[np.ndarray, pd.DataFrame]:
    model.eval()
    embeddings: List[np.ndarray] = []
    rows: List[Dict[str, Any]] = []

    for batch in loader:
        with torch.cuda.amp.autocast(
            enabled=(cfg.amp and cfg.device.startswith("cuda"))
        ):
            cell_embedding = model(batch)

        embeddings.append(cell_embedding.detach().float().cpu().numpy())
        labels = batch["labels"].cpu().numpy()

        for index in range(len(labels)):
            meta = batch["metadata"][index]
            label_id = int(labels[index])
            rows.append({
                "sample_uid": batch["sample_uids"][index],
                "source_split": batch["source_splits"][index],
                "raw_mds_index": int(batch["raw_mds_indices"][index]),
                "cell_id": batch["cell_ids"][index],
                "true_label_id": label_id,
                "true_label": id2label[label_id],
                "donor": meta.get("donor", ""),
                "method": meta.get("method", ""),
                "organ_tissue": meta.get("organ_tissue", ""),
                "cluster_label": meta.get("cluster_label", ""),
                "compartment": meta.get("compartment", ""),
                "gender": meta.get("gender", ""),
            })

    if not embeddings:
        raise RuntimeError("embedding loader 没有产生任何 batch。")

    return np.concatenate(embeddings, axis=0), pd.DataFrame(rows)


def format_cell_type_name(label: Any) -> str:
    """统一图片中的细胞类型名称：去掉下划线，并将首字母大写。"""
    cleaned = " ".join(str(label).replace("_", " ").split())
    return cleaned[:1].upper() + cleaned[1:] if cleaned else "Unknown"


def top_cell_type_ids(
    y_true: np.ndarray,
    top_n: Optional[int],
    n_classes: Optional[int] = None,
) -> List[int]:
    """按真实标签细胞数从多到少返回用于绘图的类别 ID。"""
    total_classes = int(n_cls if n_classes is None else n_classes)
    supports = np.bincount(
        np.asarray(y_true, dtype=np.int64),
        minlength=total_classes,
    )
    ordered = sorted(
        range(total_classes),
        key=lambda index: (-supports[index], index),
    )
    if top_n is None:
        return ordered
    observed = [index for index in ordered if supports[index] > 0]
    return observed[:max(1, min(int(top_n), len(observed)))]


def top_cell_type_names(
    metadata: pd.DataFrame,
    top_n: Optional[int],
) -> List[str]:
    """按真实标签细胞数选择名称，不依赖当前运行的 label2id。"""
    true_labels = metadata["true_label"].astype(str)
    supports = true_labels.value_counts().to_dict()
    ordered = sorted(supports, key=lambda label: (-supports[label], label))
    if top_n is None:
        return ordered
    return ordered[:max(1, min(int(top_n), len(ordered)))]


def save_classification_outputs(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    prefix: str,
    class_names: Optional[Sequence[str]] = None,
) -> None:
    names = (
        [str(name) for name in class_names]
        if class_names is not None
        else [id2label[index] for index in range(n_cls)]
    )
    n_classes = len(names)
    labels = list(range(n_classes))

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=names,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).T.to_csv(
        os.path.join(cfg.output_dir, f"{prefix}_classification_report.csv")
    )

    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    matrix_normalized = matrix.astype(np.float64) / np.maximum(
        matrix.sum(axis=1, keepdims=True),
        1,
    )

    pd.DataFrame(matrix, index=names, columns=names).to_csv(
        os.path.join(cfg.output_dir, f"{prefix}_confusion_matrix_raw.csv")
    )
    pd.DataFrame(matrix_normalized, index=names, columns=names).to_csv(
        os.path.join(cfg.output_dir, f"{prefix}_confusion_matrix_normalized.csv")
    )

    plot_labels = top_cell_type_ids(
        y_true,
        cfg.confusion_matrix_top_n_cell_types,
        n_classes=n_classes,
    )
    plot_names = [format_cell_type_name(names[index]) for index in plot_labels]
    matrix_plot = matrix_normalized[np.ix_(plot_labels, plot_labels)]
    n_plot = len(plot_labels)

    figure_size = max(5.6, min(9.0, 0.62 * n_plot))
    show_numbers = bool(cfg.confusion_matrix_show_values) and n_plot <= 20
    figure, axis = plt.subplots(figsize=(figure_size, figure_size), dpi=300)
    image = axis.imshow(
        matrix_plot,
        cmap="Blues",
        vmin=0,
        vmax=1,
        aspect="equal",
        interpolation="nearest",
    )
    axis.set_xticks(np.arange(n_plot))
    axis.set_yticks(np.arange(n_plot))
    axis.set_xticklabels(plot_names, rotation=45, ha="right", fontsize=8)
    axis.set_yticklabels(plot_names, fontsize=8)
    axis.set_xlabel("Predicted cell type")
    axis.set_ylabel("True cell type")
    title_suffix = f" · Top {n_plot} cell types" if n_plot < n_classes else ""
    axis.set_title(f"Confusion matrix{title_suffix}", fontsize=12, pad=8)
    for index in range(n_plot):
        axis.add_patch(
            Rectangle(
                (index - 0.5, index - 0.5),
                1,
                1,
                fill=False,
                edgecolor="#8A8A8A",
                linewidth=0.55,
            )
        )

    if show_numbers:
        for row_index in range(n_plot):
            for column_index in range(n_plot):
                value = matrix_plot[row_index, column_index]
                axis.text(
                    column_index,
                    row_index,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    color="white" if value >= 0.55 else "#333333",
                    fontsize=7.5,
                )

    colorbar = figure.colorbar(image, ax=axis, fraction=0.046, pad=0.025)
    colorbar.set_label("Fraction of true class")
    for spine in axis.spines.values():
        spine.set_visible(False)
    figure.tight_layout(pad=0.45)

    figure.savefig(
        os.path.join(cfg.output_dir, f"{prefix}_confusion_matrix.png"),
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.04,
    )
    figure.savefig(
        os.path.join(cfg.output_dir, f"{prefix}_confusion_matrix.pdf"),
        bbox_inches="tight",
        pad_inches=0.04,
    )
    plt.close(figure)


def save_per_class_recall(
    prediction_df: pd.DataFrame,
    prefix: str,
) -> pd.DataFrame:
    rows = []
    for label_id in range(n_cls):
        subset = prediction_df[prediction_df["true_label_id"] == label_id]
        support = len(subset)
        recall_value = (
            float((subset["true_label_id"] == subset["pred_label_id"]).mean())
            if support > 0
            else float("nan")
        )
        rows.append({
            "label_id": label_id,
            "cell_type": id2label[label_id],
            "support": support,
            "recall": recall_value,
        })

    recall_df = pd.DataFrame(rows).sort_values("support", ascending=False)
    recall_df.to_csv(
        os.path.join(cfg.output_dir, f"{prefix}_per_class_recall.csv"),
        index=False,
    )

    figure_width = max(7, min(18, 0.35 * len(recall_df)))
    figure, axis = plt.subplots(figsize=(figure_width, 3.8), dpi=300)
    x = np.arange(len(recall_df))
    axis.bar(x, recall_df["recall"].values)
    axis.set_ylim(0, 1.05)
    axis.set_ylabel("Recall")
    axis.set_xlabel("Cell type")
    axis.set_title(f"{dataset_name}: per-class recall")
    axis.set_xticks(x)
    recall_names = [format_cell_type_name(name) for name in recall_df["cell_type"]]
    axis.set_xticklabels(recall_names, rotation=90, fontsize=6)
    axis.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    figure.savefig(
        os.path.join(cfg.output_dir, "figures", f"{prefix}_per_class_recall.png"),
        bbox_inches="tight",
    )
    plt.close(figure)
    return recall_df


def stratified_subsample(
    embedding: np.ndarray,
    metadata: pd.DataFrame,
    max_cells: int,
    label_column: str,
    seed: int,
) -> Tuple[np.ndarray, pd.DataFrame]:
    if len(metadata) <= max_cells:
        return embedding, metadata.reset_index(drop=True)

    rng = np.random.default_rng(seed)
    selected: List[int] = []
    labels = metadata[label_column].astype(str).to_numpy()
    unique_labels = sorted(np.unique(labels))
    per_class = max(1, max_cells // max(len(unique_labels), 1))

    for label in unique_labels:
        indices = np.where(labels == label)[0]
        if len(indices) <= per_class:
            selected.extend(indices.tolist())
        else:
            selected.extend(
                rng.choice(indices, size=per_class, replace=False).tolist()
            )

    selected_array = np.asarray(sorted(selected), dtype=np.int64)
    return embedding[selected_array], metadata.iloc[selected_array].reset_index(drop=True)


def plot_true_vs_pred_umap(
    embedding: np.ndarray,
    metadata: pd.DataFrame,
    prefix: str,
) -> None:
    if not cfg.make_umap:
        return

    try:
        import umap
    except ImportError:
        print("[UMAP skipped] 请安装 umap-learn。")
        return

    plot_labels = top_cell_type_names(metadata, cfg.umap_top_n_cell_types)
    selected_mask = metadata["true_label"].astype(str).isin(plot_labels).to_numpy()
    metadata_selected = metadata.loc[selected_mask].reset_index(drop=True)
    embedding_selected = embedding[selected_mask]

    embedding_plot, metadata_plot = stratified_subsample(
        embedding_selected,
        metadata_selected,
        cfg.umap_max_cells,
        "true_label",
        cfg.seed,
    )

    reducer = umap.UMAP(
        n_neighbors=cfg.umap_n_neighbors,
        min_dist=cfg.umap_min_dist,
        metric="cosine",
        random_state=cfg.seed,
    )
    xy = reducer.fit_transform(embedding_plot)

    true_labels = metadata_plot["true_label"].astype(str).to_numpy()
    pred_labels_raw = metadata_plot["pred_label"].astype(str).to_numpy()
    labels = [label for label in plot_labels if np.any(true_labels == label)]
    other_label = "__other_cell_types__"
    pred_labels = np.where(np.isin(pred_labels_raw, labels), pred_labels_raw, other_label)
    if np.any(pred_labels == other_label):
        labels.append(other_label)


    color_pool = [
        "#D95F5F", "#4C78A8", "#59A14F", "#B279A2", "#F28E2B",
        "#76B7B2", "#E377C2", "#9C755F", "#8CD17D", "#FFBE7D",
        "#86BCB6", "#E15759", "#A0CBE8", "#D4A6C8", "#B6992D",
        "#499894", "#D37295", "#79706E", "#FABFD2", "#BAB0AC",
    ]
    color_map = {
        label: color_pool[index % len(color_pool)]
        for index, label in enumerate(labels)
    }

    color_map[other_label] = "#B8B8B8"

    figure, axes = plt.subplots(1, 2, figsize=(8.8, 3.55), dpi=300)
    for axis, values, title in [
        (axes[0], true_labels, "Annotated"),
        (axes[1], pred_labels, "Predicted"),
    ]:
        for label in labels:
            mask = values == label
            if mask.sum() == 0:
                continue
            axis.scatter(
                xy[mask, 0],
                xy[mask, 1],
                s=cfg.umap_point_size,
                alpha=cfg.umap_point_alpha,
                color=color_map[label],
                linewidths=0,
                label=(
                    "Other cell types"
                    if label == other_label
                    else format_cell_type_name(label)
                ),
            )
        if cfg.umap_show_panel_titles:
            axis.set_title(title, fontsize=9, pad=2)
        axis.set_xticks([])
        axis.set_yticks([])
        axis.margins(x=0.015, y=0.015)
        for spine in axis.spines.values():
            spine.set_visible(False)

    if cfg.umap_show_legend:
        handles, legend_labels = axes[0].get_legend_handles_labels()
        pred_handles, pred_legend_labels = axes[1].get_legend_handles_labels()
        if "Other cell types" in pred_legend_labels:
            other_index = pred_legend_labels.index("Other cell types")
            handles.append(pred_handles[other_index])
            legend_labels.append(pred_legend_labels[other_index])
        figure.legend(
            handles,
            legend_labels,
            loc="center left",
            bbox_to_anchor=(0.825, 0.5),
            title="Cell types",
            title_fontsize=8.5,
            fontsize=7.2,
            frameon=False,
            ncol=1,
            markerscale=2.4,
            handlelength=0.8,
            handletextpad=0.35,
            labelspacing=0.35,
        )
    if cfg.umap_show_suptitle:
        figure.suptitle(f"{dataset_name}: frozen encoder embeddings", fontsize=10)
    bottom_margin = 0.015
    top_margin = 0.90 if cfg.umap_show_suptitle else 0.985
    figure.subplots_adjust(
        left=0.005,
        right=0.80 if cfg.umap_show_legend else 0.995,
        top=top_margin,
        bottom=bottom_margin,
        wspace=0.055,
    )
    if cfg.umap_show_center_divider:
        figure.add_artist(
            Line2D(
                [0.5, 0.5],
                [bottom_margin + 0.12, top_margin - 0.12],
                transform=figure.transFigure,
                color="#666666",
                linewidth=0.65,
            )
        )
    figure.savefig(
        os.path.join(cfg.output_dir, "figures", f"{prefix}_umap_true_vs_pred.png"),
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.03,
    )
    figure.savefig(
        os.path.join(cfg.output_dir, "figures", f"{prefix}_umap_true_vs_pred.pdf"),
        bbox_inches="tight",
        pad_inches=0.03,
    )
    plt.close(figure)


def resolve_saved_result_path(
    base_dir: Path,
    explicit_path: str,
    candidate_names: Sequence[str],
    description: str,
) -> Path:
    """解析手工指定路径，或在结果目录内自动识别已有文件。"""
    if str(explicit_path).strip():
        path = Path(str(explicit_path).strip()).expanduser()
        if not path.is_absolute():
            path = base_dir / path
        if not path.is_file():
            raise FileNotFoundError(f"{description} 不存在：{path}")
        return path

    for name in candidate_names:
        path = base_dir / name
        if path.is_file():
            return path

    raise FileNotFoundError(
        f"在 {base_dir} 中未找到 {description}；候选文件={list(candidate_names)}。"
    )


def plot_saved_embeddings() -> Dict[str, Any]:
    """读取现有 embedding/预测表并重画 UMAP，不运行模型或重新提取表示。"""
    base_dir = Path(
        str(cfg.saved_embeddings_dir).strip() or cfg.output_dir
    ).expanduser()
    if not base_dir.is_dir():
        raise NotADirectoryError(f"已有结果目录不存在：{base_dir}")

    if cfg.evaluation_mode == "zero_shot_knn":
        embedding_candidates = (
            "zero_shot_query_embeddings.npy",
            "test_cell_embeddings.npy",
        )
        prediction_candidates = (
            "zero_shot_knn_predictions.csv",
            "test_predictions.csv",
        )
    else:
        embedding_candidates = (
            "test_cell_embeddings.npy",
            "zero_shot_query_embeddings.npy",
        )
        prediction_candidates = (
            "test_predictions.csv",
            "zero_shot_knn_predictions.csv",
        )

    embedding_path = resolve_saved_result_path(
        base_dir,
        cfg.saved_embedding_path,
        embedding_candidates,
        "embedding .npy 文件",
    )
    predictions_path = resolve_saved_result_path(
        base_dir,
        cfg.saved_predictions_path,
        prediction_candidates,
        "predictions CSV 文件",
    )

    embedding = np.load(embedding_path, mmap_mode="r", allow_pickle=False)
    metadata = pd.read_csv(predictions_path)
    if embedding.ndim != 2:
        raise ValueError(
            f"embedding 必须为二维数组，实际 shape={embedding.shape}。"
        )
    if len(metadata) != embedding.shape[0]:
        raise ValueError(
            "embedding 行数与 predictions CSV 行数不一致："
            f"{embedding.shape[0]} vs {len(metadata)}。"
        )

    required_columns = {"true_label", "pred_label"}
    missing_columns = sorted(required_columns - set(metadata.columns))
    if missing_columns:
        raise KeyError(
            f"predictions CSV 缺少绘图字段：{missing_columns}。"
        )
    metadata = metadata.copy()
    metadata["true_label"] = metadata["true_label"].fillna("Unknown").astype(str)
    metadata["pred_label"] = metadata["pred_label"].fillna("Unknown").astype(str)

    output_dir = Path(
        str(cfg.saved_plot_output_dir).strip() or base_dir
    ).expanduser()
    (output_dir / "figures").mkdir(parents=True, exist_ok=True)
    prefix = str(cfg.saved_plot_prefix).strip() or embedding_path.stem

    original_output_dir = cfg.output_dir
    try:
        cfg.output_dir = str(output_dir)
        class_names = sorted(
            set(metadata["true_label"]) | set(metadata["pred_label"])
        )
        label_to_id = {
            label: index for index, label in enumerate(class_names)
        }
        redraw_true_ids = metadata["true_label"].map(label_to_id).to_numpy()
        redraw_pred_ids = metadata["pred_label"].map(label_to_id).to_numpy()
        save_classification_outputs(
            redraw_true_ids,
            redraw_pred_ids,
            prefix=prefix,
            class_names=class_names,
        )
        plot_true_vs_pred_umap(
            embedding,
            metadata,
            prefix=prefix,
        )
    finally:
        cfg.output_dir = original_output_dir

    result = {
        "mode": "redraw_saved_embeddings",
        "embedding_path": str(embedding_path),
        "predictions_path": str(predictions_path),
        "output_dir": str(output_dir),
        "plot_prefix": prefix,
        "n_cells": int(embedding.shape[0]),
        "embedding_dim": int(embedding.shape[1]),
        "n_cell_types": int(metadata["true_label"].nunique()),
        "confusion_matrix_prefix": prefix,
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


def append_summary(summary_row: Dict[str, Any]) -> str:
    summary_path = os.path.join(cfg.output_root, "summary_results.csv")
    new_frame = pd.DataFrame([summary_row])
    if os.path.exists(summary_path):
        old_frame = pd.read_csv(summary_path)
        all_frame = pd.concat([old_frame, new_frame], ignore_index=True, sort=False)
    else:
        all_frame = new_frame
    all_frame.to_csv(summary_path, index=False)
    return summary_path


In [ ]:




def create_or_load_zero_shot_manifest(
    metadata: pd.DataFrame,
) -> pd.DataFrame:
    required_columns = {
        "sample_uid",
        "source_split",
        "raw_mds_index",
        "cell_id",
        "true_label",
        "true_label_id",
    }
    missing = required_columns - set(metadata.columns)
    if missing:
        raise KeyError(
            f"构建 split manifest 缺少字段：{sorted(missing)}"
        )

    if metadata["sample_uid"].duplicated().any():
        duplicates = metadata.loc[
            metadata["sample_uid"].duplicated(),
            "sample_uid",
        ].head(20).tolist()
        raise ValueError(f"sample_uid 不唯一：{duplicates}")

    manifest_path = Path(cfg.zero_shot_manifest_path)

    if cfg.zero_shot_reuse_manifest and manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        required_manifest = {
            "sample_uid",
            "cell_type",
            "zero_shot_role",
            "split_mode",
        }
        missing_manifest = required_manifest - set(manifest.columns)
        if missing_manifest:
            raise ValueError(
                f"已有 manifest 缺少字段：{sorted(missing_manifest)}"
            )

        if not manifest["split_mode"].astype(str).eq(
            cfg.zero_shot_split_mode
        ).all():
            raise ValueError(
                "已有 manifest 的 split_mode 与当前配置不一致。"
            )

        current_uids = set(metadata["sample_uid"].astype(str))
        manifest_uids = set(manifest["sample_uid"].astype(str))
        if current_uids != manifest_uids:
            only_current = sorted(current_uids - manifest_uids)[:20]
            only_manifest = sorted(manifest_uids - current_uids)[:20]
            raise ValueError(
                "已有共享 manifest 与当前数据池不匹配。"
                f"\nonly_current={only_current}"
                f"\nonly_manifest={only_manifest}"
            )

        label_check = metadata[["sample_uid", "true_label"]].merge(
            manifest[["sample_uid", "cell_type"]],
            on="sample_uid",
            how="left",
            validate="one_to_one",
        )
        mismatch = label_check[
            label_check["true_label"].astype(str)
            != label_check["cell_type"].astype(str)
        ]
        if len(mismatch) > 0:
            raise ValueError(
                "共享 manifest 的 cell_type 与当前数据不一致。"
            )

        print("[split manifest] reused:", manifest_path)
        return manifest

    manifest = metadata[
        [
            "sample_uid",
            "source_split",
            "raw_mds_index",
            "cell_id",
            "true_label",
            "true_label_id",
        ]
    ].copy()
    manifest = manifest.rename(
        columns={"true_label": "cell_type"}
    )

    if cfg.zero_shot_split_mode == "existing_split":
        reference_sources = {cfg.zero_shot_reference_split}
        if cfg.zero_shot_use_val_as_reference:
            reference_sources.add("val")

        roles = np.full(len(metadata), "unused", dtype=object)
        source_values = metadata["source_split"].astype(str)
        roles[source_values.isin(reference_sources).to_numpy()] = "reference"
        roles[
            source_values.eq(cfg.zero_shot_query_split).to_numpy()
        ] = "query"

        n_reference = int(np.sum(roles == "reference"))
        n_query = int(np.sum(roles == "query"))
        n_unused = int(np.sum(roles == "unused"))

        if n_reference == 0:
            raise ValueError(
                "existing_split 模式没有 reference cells；"
                f"reference split={cfg.zero_shot_reference_split!r}"
            )
        if n_query == 0:
            raise ValueError(
                "existing_split 模式没有 query cells；"
                f"query split={cfg.zero_shot_query_split!r}"
            )
        if n_unused > 0:
            unused_splits = sorted(
                metadata.loc[
                    roles == "unused",
                    "source_split",
                ].astype(str).unique()
            )
            raise ValueError(
                "当前提取的数据包含未分配的 split："
                f"{unused_splits}"
            )

        manifest["zero_shot_role"] = roles
        manifest["split_mode"] = "existing_split"
        manifest["reference_split"] = cfg.zero_shot_reference_split
        manifest["query_split"] = cfg.zero_shot_query_split
        manifest["val_used_as_reference"] = bool(
            cfg.zero_shot_use_val_as_reference
        )
        manifest["split_seed"] = np.nan
        manifest["stratified"] = False
        manifest["reference_ratio"] = float(
            n_reference / (n_reference + n_query)
        )

    elif cfg.zero_shot_split_mode == "resplit_all":
        class_counts = metadata["true_label"].value_counts()
        too_rare = class_counts[class_counts < 2]
        if cfg.zero_shot_stratify and len(too_rare) > 0:
            raise ValueError(
                "分层重划分要求每个 cell type 至少有 2 个细胞；"
                f"过少类别={too_rare.to_dict()}"
            )

        indices = np.arange(len(metadata))
        stratify_labels = (
            metadata["true_label_id"].to_numpy()
            if cfg.zero_shot_stratify
            else None
        )
        reference_indices, query_indices = train_test_split(
            indices,
            train_size=cfg.zero_shot_reference_ratio,
            test_size=1.0 - cfg.zero_shot_reference_ratio,
            stratify=stratify_labels,
            random_state=cfg.zero_shot_split_seed,
            shuffle=True,
        )

        roles = np.full(len(metadata), "", dtype=object)
        roles[reference_indices] = "reference"
        roles[query_indices] = "query"

        manifest["zero_shot_role"] = roles
        manifest["split_mode"] = "resplit_all"
        manifest["reference_split"] = "pooled"
        manifest["query_split"] = "pooled"
        manifest["val_used_as_reference"] = False
        manifest["split_seed"] = int(cfg.zero_shot_split_seed)
        manifest["stratified"] = bool(cfg.zero_shot_stratify)
        manifest["reference_ratio"] = float(
            cfg.zero_shot_reference_ratio
        )

    else:
        raise ValueError(
            "zero_shot_split_mode 仅支持 "
            "existing_split 或 resplit_all。"
        )

    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(manifest_path, index=False)
    print("[split manifest] created:", manifest_path)
    return manifest


def get_zero_shot_active_splits() -> List[str]:
    if cfg.zero_shot_split_mode == "existing_split":
        split_names: List[str] = [
            cfg.zero_shot_reference_split,
        ]
        if (
            cfg.zero_shot_use_val_as_reference
            and "val" not in split_names
            and cfg.zero_shot_query_split != "val"
        ):
            split_names.append("val")
        if cfg.zero_shot_query_split not in split_names:
            split_names.append(cfg.zero_shot_query_split)
    else:
        split_names = list(cfg.zero_shot_pool_splits)


    return list(dict.fromkeys(split_names))

def run_zero_shot_knn() -> Dict[str, Any]:
    print("=" * 88)
    print("FROZEN-EMBEDDING 5-NN ZERO-SHOT")
    print("=" * 88)

    model = FrozenEmbeddingModel(cfg, encoder_state).to(cfg.device)
    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    if trainable_parameters != 0:
        raise AssertionError("zero-shot 模式下 encoder 存在可训练参数。")

    all_embeddings: List[np.ndarray] = []
    all_metadata: List[pd.DataFrame] = []

    active_splits = get_zero_shot_active_splits()
    print("active splits        :", active_splits)

    for split_name in active_splits:
        loader = make_loader(datasets[split_name], shuffle=False)
        split_embedding, split_metadata = extract_frozen_embeddings(model, loader)
        all_embeddings.append(split_embedding)
        all_metadata.append(split_metadata)
        print(f"[embedding] {split_name}: {split_embedding.shape}")

    embedding = np.concatenate(all_embeddings, axis=0)
    metadata = pd.concat(all_metadata, ignore_index=True)

    if len(metadata) != embedding.shape[0]:
        raise RuntimeError("embedding 行数与 metadata 行数不一致。")

    manifest = create_or_load_zero_shot_manifest(metadata)
    metadata_with_role = metadata.merge(
        manifest[["sample_uid", "zero_shot_role"]],
        on="sample_uid",
        how="left",
        validate="one_to_one",
    )

    if metadata_with_role["zero_shot_role"].isna().any():
        raise RuntimeError("部分细胞没有 zero_shot_role。")


    metadata_with_role[
        [
            "sample_uid",
            "source_split",
            "raw_mds_index",
            "cell_id",
            "true_label",
            "true_label_id",
            "zero_shot_role",
        ]
    ].to_csv(
        os.path.join(cfg.output_dir, "zero_shot_knn_split_manifest.csv"),
        index=False,
    )

    reference_mask = metadata_with_role["zero_shot_role"].eq("reference").to_numpy()
    query_mask = metadata_with_role["zero_shot_role"].eq("query").to_numpy()

    reference_embedding = embedding[reference_mask]
    query_embedding = embedding[query_mask]
    reference_metadata = metadata_with_role.loc[reference_mask].reset_index(drop=True)
    query_metadata = metadata_with_role.loc[query_mask].reset_index(drop=True)

    reference_labels = reference_metadata["true_label_id"].to_numpy(dtype=np.int64)
    query_labels = query_metadata["true_label_id"].to_numpy(dtype=np.int64)

    reference_classes = set(reference_labels.tolist())
    query_classes = set(query_labels.tolist())
    unsupported_classes = sorted(query_classes - reference_classes)
    if unsupported_classes:
        unsupported_names = [id2label[class_id] for class_id in unsupported_classes]
        raise RuntimeError(
            "query 中存在 reference 未覆盖的类别："
            f"{unsupported_names}。请检查 train-seen 标签过滤或划分配置。"
        )

    if len(reference_embedding) < cfg.zero_shot_k:
        raise ValueError(
            f"reference cells={len(reference_embedding)} 小于 k={cfg.zero_shot_k}。"
        )

    knn = KNeighborsClassifier(
        n_neighbors=cfg.zero_shot_k,
        weights=cfg.zero_shot_weights,
        metric=cfg.zero_shot_metric,
        p=cfg.zero_shot_p,
        algorithm="auto",
        n_jobs=-1,
    )
    knn.fit(reference_embedding, reference_labels)

    query_predictions = knn.predict(query_embedding).astype(np.int64)
    query_probabilities = knn.predict_proba(query_embedding)
    neighbor_distances, neighbor_indices = knn.kneighbors(
        query_embedding,
        n_neighbors=cfg.zero_shot_k,
        return_distance=True,
    )

    metrics = compute_metrics(query_labels, query_predictions, n_cls)
    metrics.update({
        "evaluation_mode": (
            "predefined_split_knn_zero_shot"
            if cfg.zero_shot_split_mode == "existing_split"
            else "same_dataset_knn_zero_shot"
        ),
        "protocol": (
            "existing_train_reference_test_query_frozen_embedding_5nn"
            if cfg.zero_shot_split_mode == "existing_split"
            else "fixed_stratified_resplit_frozen_embedding_5nn"
        ),
        "classifier": "KNeighborsClassifier",
        "n_neighbors": int(cfg.zero_shot_k),
        "weights": str(cfg.zero_shot_weights),
        "metric": str(cfg.zero_shot_metric),
        "p": int(cfg.zero_shot_p),
        "embedding_normalization": "none",
        "split_mode": str(cfg.zero_shot_split_mode),
        "reference_split": (
            cfg.zero_shot_reference_split
            if cfg.zero_shot_split_mode == "existing_split"
            else "pooled"
        ),
        "query_split": (
            cfg.zero_shot_query_split
            if cfg.zero_shot_split_mode == "existing_split"
            else "pooled"
        ),
        "val_used_as_reference": bool(
            cfg.zero_shot_use_val_as_reference
            if cfg.zero_shot_split_mode == "existing_split"
            else False
        ),
        "reference_ratio": float(
            len(reference_metadata)
            / max(len(reference_metadata) + len(query_metadata), 1)
        ),
        "split_seed": (
            None
            if cfg.zero_shot_split_mode == "existing_split"
            else int(cfg.zero_shot_split_seed)
        ),
        "stratified_split": bool(
            False
            if cfg.zero_shot_split_mode == "existing_split"
            else cfg.zero_shot_stratify
        ),
        "n_total_cells": int(len(metadata_with_role)),
        "n_reference_cells": int(len(reference_metadata)),
        "n_query_cells": int(len(query_metadata)),
        "n_cell_types": int(n_cls),
        "reference_label_coverage": float(
            len(query_classes & reference_classes) / max(len(query_classes), 1)
        ),
        "encoder_trainable_parameters": int(trainable_parameters),
        "classifier_trainable_parameters": 0,
        "loss": None,
        "selected_epoch": None,
        "checkpoint_selection": None,
        "shared_manifest_path": str(cfg.zero_shot_manifest_path),
    })

    class_to_probability_column = {
        int(class_id): column_index
        for column_index, class_id in enumerate(knn.classes_)
    }

    prediction_df = query_metadata.copy()
    prediction_df["pred_label_id"] = query_predictions
    prediction_df["pred_label"] = [id2label[int(value)] for value in query_predictions]
    prediction_df["correct"] = query_labels == query_predictions
    prediction_df["max_vote_fraction"] = query_probabilities.max(axis=1)
    prediction_df["true_label_vote_fraction"] = [
        float(query_probabilities[row_index, class_to_probability_column[int(label_id)]])
        for row_index, label_id in enumerate(query_labels)
    ]
    prediction_df["nearest_neighbor_distance"] = neighbor_distances[:, 0]
    prediction_df["mean_neighbor_distance"] = neighbor_distances.mean(axis=1)

    for neighbor_rank in range(cfg.zero_shot_k):
        reference_rows = neighbor_indices[:, neighbor_rank]
        neighbor_label_ids = reference_labels[reference_rows]
        prediction_df[f"neighbor_{neighbor_rank + 1}_label_id"] = neighbor_label_ids
        prediction_df[f"neighbor_{neighbor_rank + 1}_label"] = [
            id2label[int(value)] for value in neighbor_label_ids
        ]
        prediction_df[f"neighbor_{neighbor_rank + 1}_distance"] = (
            neighbor_distances[:, neighbor_rank]
        )
        prediction_df[f"neighbor_{neighbor_rank + 1}_sample_uid"] = (
            reference_metadata.iloc[reference_rows]["sample_uid"].to_numpy()
        )

    with open(
        os.path.join(cfg.output_dir, "zero_shot_knn_metrics.json"),
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(metrics, handle, ensure_ascii=False, indent=2)

    prediction_df.to_csv(
        os.path.join(cfg.output_dir, "zero_shot_knn_predictions.csv"),
        index=False,
    )
    reference_metadata.to_csv(
        os.path.join(cfg.output_dir, "zero_shot_reference_metadata.csv"),
        index=False,
    )
    query_metadata.to_csv(
        os.path.join(cfg.output_dir, "zero_shot_query_metadata.csv"),
        index=False,
    )

    if cfg.save_cell_emb:
        np.save(
            os.path.join(cfg.output_dir, "zero_shot_reference_embeddings.npy"),
            reference_embedding,
        )
        np.save(
            os.path.join(cfg.output_dir, "zero_shot_query_embeddings.npy"),
            query_embedding,
        )

    save_classification_outputs(
        query_labels,
        query_predictions,
        prefix="zero_shot_knn",
    )
    save_per_class_recall(prediction_df, prefix="zero_shot_knn")
    plot_true_vs_pred_umap(
        query_embedding,
        prediction_df,
        prefix="zero_shot_knn_query",
    )

    summary_row = {
        "run_name": cfg.run_name,
        "dataset_name": dataset_name,
        "model_name": cfg.model_name,
        "output_dir": cfg.output_dir,
        "evaluation_mode": "zero_shot_knn",
        "protocol": (
            "existing_train_reference_test_query_5nn"
            if cfg.zero_shot_split_mode == "existing_split"
            else "fixed_stratified_resplit_same_dataset_5nn"
        ),
        "encoder_ckpt": cfg.encoder_ckpt,
        "vocab_size": cfg.vocab_size,
        "hidden_dim": cfg.hidden_dim,
        "nlayers": cfg.nlayers,
        "nhead": cfg.nhead,
        "value_mode": cfg.value_mode,
        "max_len": cfg.max_len,
        "finetune_mode": "none",
        "head_type": "none",
        "classifier": "5-NN",
        "k": cfg.zero_shot_k,
        "split_mode": cfg.zero_shot_split_mode,
        "reference_split": (
            cfg.zero_shot_reference_split
            if cfg.zero_shot_split_mode == "existing_split"
            else "pooled"
        ),
        "query_split": (
            cfg.zero_shot_query_split
            if cfg.zero_shot_split_mode == "existing_split"
            else "pooled"
        ),
        "val_used_as_reference": (
            cfg.zero_shot_use_val_as_reference
            if cfg.zero_shot_split_mode == "existing_split"
            else False
        ),
        "reference_ratio": metrics["reference_ratio"],
        "split_seed": metrics["split_seed"],
        "stratified": metrics["stratified_split"],
        "Accuracy": metrics["accuracy"],
        "Micro-F1": metrics["micro_f1"],
        "Macro-F1": metrics["macro_f1"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "weighted_f1": metrics["weighted_f1"],
        "test_loss": np.nan,
        "n_reference_cells": metrics["n_reference_cells"],
        "n_query_cells": metrics["n_query_cells"],
        "n_cell_types": metrics["n_cell_types"],
        "encoder_trainable_params": 0,
        "classifier_trainable_params": 0,
        "selected_epoch": np.nan,
        "checkpoint_path": np.nan,
    }
    summary_path = append_summary(summary_row)

    print("=" * 88)
    print("FROZEN-EMBEDDING 5-NN ZERO-SHOT RESULTS")
    print("split mode           :", cfg.zero_shot_split_mode)
    print("reference split      :", metrics["reference_split"])
    print("query split          :", metrics["query_split"])
    print("encoder frozen       : True")
    print("trainable parameters : 0")
    print("total cells          :", metrics["n_total_cells"])
    print("reference cells      :", metrics["n_reference_cells"])
    print("query cells          :", metrics["n_query_cells"])
    print("cell types           :", metrics["n_cell_types"])
    print("reference coverage   :", f"{metrics['reference_label_coverage']:.4f}")
    print("Accuracy             :", f"{metrics['accuracy']:.6f}")
    print("Micro-F1             :", f"{metrics['micro_f1']:.6f}")
    print("Macro-F1             :", f"{metrics['macro_f1']:.6f}")
    print("Macro-Precision      :", f"{metrics['precision']:.6f}")
    print("Macro-Recall         :", f"{metrics['recall']:.6f}")
    print("Weighted-F1          :", f"{metrics['weighted_f1']:.6f}")
    print("No neural-network training was performed.")
    print("No downstream checkpoint was selected.")
    print("saved to             :", cfg.output_dir)
    print("summary saved to     :", summary_path)
    print("=" * 88)

    return metrics


In [ ]:




def run_supervised() -> Dict[str, Any]:
    train_loader = make_loader(datasets["train"], shuffle=True)
    val_loader = make_loader(datasets["val"], shuffle=False)
    test_loader = make_loader(datasets["test"], shuffle=False)

    model = AnnotationModel(cfg, encoder_state, n_cls).to(cfg.device)

    if cfg.finetune_mode == "linear_probe":
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False
        for parameter in model.classifier.parameters():
            parameter.requires_grad = True
        model.encoder.eval()
        optimizer = torch.optim.AdamW(
            model.classifier.parameters(),
            lr=cfg.lr_head,
            weight_decay=cfg.weight_decay,
        )
    else:
        for parameter in model.parameters():
            parameter.requires_grad = True
        optimizer = torch.optim.AdamW(
            [
                {"params": model.encoder.parameters(), "lr": cfg.lr_encoder},
                {"params": model.classifier.parameters(), "lr": cfg.lr_head},
            ],
            weight_decay=cfg.weight_decay,
        )

    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(
        enabled=(cfg.amp and cfg.device.startswith("cuda"))
    )

    def train_one_epoch() -> Dict[str, float]:
        model.train()
        if cfg.finetune_mode == "linear_probe":
            model.encoder.eval()
            model.classifier.train()

        losses: List[float] = []
        all_true: List[int] = []
        all_pred: List[int] = []

        for batch in train_loader:
            labels = batch["labels"].to(cfg.device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(
                enabled=(cfg.amp and cfg.device.startswith("cuda"))
            ):
                output = model(batch)
                loss = criterion(output["logits"], labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            losses.append(float(loss.detach().cpu()))
            all_true.extend(labels.detach().cpu().numpy().tolist())
            all_pred.extend(
                output["logits"].argmax(dim=1).detach().cpu().numpy().tolist()
            )

        metrics = compute_metrics(
            np.asarray(all_true),
            np.asarray(all_pred),
            n_cls,
        )
        metrics["loss"] = float(np.mean(losses))
        return metrics

    @torch.no_grad()
    def evaluate(
        loader: DataLoader,
        return_details: bool,
    ) -> Tuple[Dict[str, float], Optional[pd.DataFrame], Optional[np.ndarray]]:
        model.eval()
        losses: List[float] = []
        all_true: List[int] = []
        all_pred: List[int] = []
        all_max_prob: List[float] = []
        all_true_prob: List[float] = []
        all_embeddings: List[np.ndarray] = []
        rows: List[Dict[str, Any]] = []

        for batch in loader:
            labels = batch["labels"].to(cfg.device)
            with torch.cuda.amp.autocast(
                enabled=(cfg.amp and cfg.device.startswith("cuda"))
            ):
                output = model(batch)
                logits = output["logits"]
                loss = criterion(logits, labels)

            probabilities = torch.softmax(logits, dim=1)
            predictions = logits.argmax(dim=1)
            losses.append(float(loss.detach().cpu()))
            labels_np = labels.detach().cpu().numpy()
            predictions_np = predictions.detach().cpu().numpy()
            all_true.extend(labels_np.tolist())
            all_pred.extend(predictions_np.tolist())
            all_max_prob.extend(probabilities.max(dim=1).values.cpu().numpy().tolist())
            all_true_prob.extend(
                probabilities.gather(1, labels.view(-1, 1)).squeeze(1).cpu().numpy().tolist()
            )
            all_embeddings.append(output["cell_emb"].detach().float().cpu().numpy())

            if return_details:
                for index in range(len(labels_np)):
                    meta = batch["metadata"][index]
                    rows.append({
                        "sample_uid": batch["sample_uids"][index],
                        "source_split": batch["source_splits"][index],
                        "raw_mds_index": int(batch["raw_mds_indices"][index]),
                        "cell_id": batch["cell_ids"][index],
                        "true_label_id": int(labels_np[index]),
                        "pred_label_id": int(predictions_np[index]),
                        "true_label": id2label[int(labels_np[index])],
                        "pred_label": id2label[int(predictions_np[index])],
                        "correct": bool(labels_np[index] == predictions_np[index]),
                        "max_prob": float(probabilities.max(dim=1).values[index].cpu()),
                        "true_prob": float(
                            probabilities[index, int(labels_np[index])].cpu()
                        ),
                        "donor": meta.get("donor", ""),
                        "method": meta.get("method", ""),
                        "organ_tissue": meta.get("organ_tissue", ""),
                        "cluster_label": meta.get("cluster_label", ""),
                        "compartment": meta.get("compartment", ""),
                        "gender": meta.get("gender", ""),
                    })

        metrics = compute_metrics(
            np.asarray(all_true),
            np.asarray(all_pred),
            n_cls,
        )
        metrics["loss"] = float(np.mean(losses))
        metrics["n_cells"] = int(len(all_true))
        metrics["n_cell_types"] = int(n_cls)
        detail_df = pd.DataFrame(rows) if return_details else None
        embedding = np.concatenate(all_embeddings, axis=0) if all_embeddings else None
        return metrics, detail_df, embedding

    best_perf = -float("inf")
    best_perf_epoch = -1
    best_val_loss = float("inf")
    best_loss_epoch = -1
    last_epoch = -1
    bad_epochs = 0
    train_log: List[Dict[str, Any]] = []

    best_perf_path = os.path.join(cfg.output_dir, "annotation_best_by_perf.pt")
    best_loss_path = os.path.join(cfg.output_dir, "annotation_best_by_loss.pt")
    last_path = os.path.join(cfg.output_dir, "annotation_last.pt")

    def make_checkpoint(
        epoch: int,
        selection_mode: str,
        selection_metric: str,
        selection_value: float,
        val_metrics: Dict[str, float],
    ) -> Dict[str, Any]:
        return {
            "model_state_dict": model.state_dict(),
            "cfg": asdict(cfg),
            "label2id": label2id,
            "id2label": id2label,
            "epoch": int(epoch),
            "selection_mode": str(selection_mode),
            "selection_metric": str(selection_metric),
            "selection_value": float(selection_value),
            "val_metrics": {str(k): float(v) for k, v in val_metrics.items()},
        }

    for epoch in range(1, cfg.epochs + 1):
        start_time = time.time()
        train_metrics = train_one_epoch()
        val_metrics, _, _ = evaluate(val_loader, return_details=False)
        current_perf = float(val_metrics[cfg.metric_for_best])
        current_loss = float(val_metrics["loss"])

        row = {"epoch": epoch, "time_sec": round(time.time() - start_time, 2)}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        train_log.append(row)
        pd.DataFrame(train_log).to_csv(
            os.path.join(cfg.output_dir, "train_log.csv"),
            index=False,
        )

        print(
            f"Epoch {epoch:03d} | train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={current_loss:.4f} | val_acc={val_metrics['accuracy']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        last_epoch = epoch
        torch.save(
            make_checkpoint(epoch, "last", "epoch", float(epoch), val_metrics),
            last_path,
        )

        if current_perf > best_perf:
            best_perf = current_perf
            best_perf_epoch = epoch
            bad_epochs = 0
            torch.save(
                make_checkpoint(
                    epoch,
                    "perf",
                    cfg.metric_for_best,
                    current_perf,
                    val_metrics,
                ),
                best_perf_path,
            )
        else:
            bad_epochs += 1

        if current_loss < best_val_loss:
            best_val_loss = current_loss
            best_loss_epoch = epoch
            torch.save(
                make_checkpoint(epoch, "loss", "loss", current_loss, val_metrics),
                best_loss_path,
            )

        if bad_epochs >= cfg.patience:
            print("Early stopping triggered.")
            break

    checkpoint_map = {
        "perf": best_perf_path,
        "loss": best_loss_path,
        "last": last_path,
    }
    selected_checkpoint = checkpoint_map[cfg.select_ckpt_for_test]
    checkpoint = torch.load(selected_checkpoint, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(cfg.device)

    test_metrics, prediction_df, test_embedding = evaluate(
        test_loader,
        return_details=True,
    )
    assert prediction_df is not None
    assert test_embedding is not None

    test_metrics_to_save = {
        **test_metrics,
        "select_ckpt_for_test": cfg.select_ckpt_for_test,
        "checkpoint_path": selected_checkpoint,
        "selected_epoch": int(checkpoint.get("epoch", -1)),
        "selection_metric": checkpoint.get("selection_metric", "unknown"),
        "selection_value": checkpoint.get("selection_value", float("nan")),
    }

    with open(os.path.join(cfg.output_dir, "test_metrics.json"), "w", encoding="utf-8") as handle:
        json.dump(test_metrics_to_save, handle, ensure_ascii=False, indent=2)

    prediction_df.to_csv(
        os.path.join(cfg.output_dir, "test_predictions.csv"),
        index=False,
    )
    if cfg.save_cell_emb:
        np.save(os.path.join(cfg.output_dir, "test_cell_embeddings.npy"), test_embedding)

    save_classification_outputs(
        prediction_df["true_label_id"].to_numpy(),
        prediction_df["pred_label_id"].to_numpy(),
        prefix="test",
    )
    save_per_class_recall(prediction_df, prefix="test")
    plot_true_vs_pred_umap(test_embedding, prediction_df, prefix="test")

    encoder_trainable = sum(
        parameter.numel()
        for parameter in model.encoder.parameters()
        if parameter.requires_grad
    )
    classifier_trainable = sum(
        parameter.numel()
        for parameter in model.classifier.parameters()
        if parameter.requires_grad
    )

    summary_row = {
        "run_name": cfg.run_name,
        "dataset_name": dataset_name,
        "model_name": cfg.model_name,
        "output_dir": cfg.output_dir,
        "evaluation_mode": "supervised",
        "encoder_ckpt": cfg.encoder_ckpt,
        "vocab_size": cfg.vocab_size,
        "hidden_dim": cfg.hidden_dim,
        "nlayers": cfg.nlayers,
        "nhead": cfg.nhead,
        "value_mode": cfg.value_mode,
        "max_len": cfg.max_len,
        "finetune_mode": cfg.finetune_mode,
        "head_type": cfg.head_type,
        "Accuracy": test_metrics["accuracy"],
        "Micro-F1": test_metrics["micro_f1"],
        "Macro-F1": test_metrics["macro_f1"],
        "Precision": test_metrics["precision"],
        "Recall": test_metrics["recall"],
        "weighted_f1": test_metrics["weighted_f1"],
        "test_loss": test_metrics["loss"],
        "n_test_cells": test_metrics["n_cells"],
        "n_cell_types": test_metrics["n_cell_types"],
        "encoder_trainable_params": encoder_trainable,
        "classifier_trainable_params": classifier_trainable,
        "selected_epoch": int(checkpoint.get("epoch", -1)),
        "checkpoint_path": selected_checkpoint,
        "best_perf_epoch": best_perf_epoch,
        "best_perf_value": best_perf,
        "best_loss_epoch": best_loss_epoch,
        "best_val_loss": best_val_loss,
        "last_epoch": last_epoch,
    }
    summary_path = append_summary(summary_row)

    print("=" * 88)
    print("SUPERVISED TEST RESULTS")
    print(json.dumps(test_metrics_to_save, ensure_ascii=False, indent=2))
    print("saved to         :", cfg.output_dir)
    print("summary saved to :", summary_path)
    print("=" * 88)
    return test_metrics_to_save


In [ ]:




if cfg.redraw_saved_embeddings:
    final_metrics = plot_saved_embeddings()
elif cfg.evaluation_mode == "zero_shot_knn":
    final_metrics = run_zero_shot_knn()
else:
    final_metrics = run_supervised()

final_metrics


## Zero-shot 输出说明

改进版 5-NN 模式会生成：

- `zero_shot_knn_metrics.json`：Accuracy、Micro/Macro-F1、Macro-Precision/Recall 等；
- `zero_shot_knn_predictions.csv`：每个 query cell 的预测、5 个邻居、距离和投票比例；
- `zero_shot_knn_split_manifest.csv`：本次 reference/query 角色；
- dataset 级共享 manifest：供不同模型复用完全相同的划分；
- classification report、混淆矩阵、per-class recall；
- reference/query embeddings（`save_cell_emb=True` 时）；
- query UMAP true-vs-pred（安装 `umap-learn` 且 `make_umap=True` 时）；
- `summary_results.csv` 中追加一行结果。

Zero-shot 分支不会产生训练 loss、epoch、optimizer 或下游 checkpoint。
